In [21]:
import pandas as pd
import numpy as np

In [ ]:
SURVEY_FILE = "australia.csv"
POLICY_FILE = "OxCGRT_AUS_latest.csv"

# Threshold for removing variables with excessive missing values
MISSING_RATE_THRESHOLD = 0.21
# Window size used to smooth policy intensity over time
ROLLING_WINDOW_DAYS = 14
# Policy intensity threshold used to define the start of mandate period
MANDATE_THRESHOLD = 3

# Time window during which survey consent rules apply
CONSENT_START = "2021-02-10"
CONSENT_END = "2021-10-18"

In [ ]:
# Load raw data
survey_raw = pd.read_csv(
    SURVEY_FILE,
    na_values=[" ", "__NA__"],
    keep_default_na=True,
    low_memory=False
)

policy_raw = pd.read_csv(POLICY_FILE)

In [ ]:
# Identify mandate start dates from policy data
# Keep only the policy variables needed for facial covering mandates
policy_df = policy_raw.loc[
    :,
    ["RegionName", "RegionCode", "Date", "H6M_Facial Coverings"]
].copy()

policy_df["Date"] = pd.to_datetime(policy_df["Date"], format="%Y%m%d", errors="coerce")
policy_df = policy_df.dropna(subset=["RegionName", "Date"]).copy()
policy_df = policy_df.sort_values(["RegionName", "Date"])

# Calculate 14-day rolling mean of facial covering policy score by state
policy_rolling = (
    policy_df
    .set_index("Date")
    .groupby("RegionName")["H6M_Facial Coverings"]
    .rolling(window=ROLLING_WINDOW_DAYS, min_periods=ROLLING_WINDOW_DAYS)
    .mean()
    .reset_index()
    .rename(columns={"H6M_Facial Coverings": "H6M_rolling_mean"})
)

# Define mandate start date as the first date when rolling policy score reaches the threshold
mandate_start_df = (
    policy_rolling.loc[policy_rolling["H6M_rolling_mean"] >= MANDATE_THRESHOLD]
    .groupby("RegionName", as_index=False)
    .first()
    .rename(columns={"Date": "mandate_start_date"})
)

mandate_start_df.to_csv("mandate_start_dates.csv", index=False)

In [ ]:
# Clean survey data
survey_df = survey_raw.copy()

# Convert survey end date into datetime format
survey_df["endtime"] = pd.to_datetime(
    survey_df["endtime"].astype(str).str.split().str[0],
    format="%d/%m/%Y",
    errors="coerce"
)

survey_df = survey_df.dropna(subset=["endtime"]).copy()

In [ ]:
# Remove variables with high missing rates
missing_rate = survey_df.isna().mean()
high_missing_cols = missing_rate[missing_rate > MISSING_RATE_THRESHOLD].index.tolist()

survey_df = survey_df.drop(columns=high_missing_cols)

missing_summary = pd.DataFrame({
    "Variable Name": missing_rate.index,
    "Missing Rate": missing_rate.values
})
missing_summary.to_csv("missing_value_summary.csv", index=False)

In [ ]:
# Handle missing values during the consent period
consent_mask = (
    (survey_df["endtime"] >= pd.to_datetime(CONSENT_START)) &
    (survey_df["endtime"] <= pd.to_datetime(CONSENT_END))
)

phq_cols = [f"PHQ4_{i}" for i in range(1, 5)]
d1_health_cols = [f"d1_health_{i}" for i in range(1, 14)] + [f"d1_health_{i}" for i in range(98, 100)]

for col in phq_cols:
    if col in survey_df.columns:
        survey_df.loc[consent_mask, col] = survey_df.loc[consent_mask, col].fillna("N/A")

for col in d1_health_cols:
    if col in survey_df.columns:
        survey_df.loc[consent_mask, col] = survey_df.loc[consent_mask, col].fillna("N/A")

In [ ]:
# Construct behavioural outcome variables
# Convert frequency responses into numeric scores
frequency_map = {
    "Always": 5,
    "Frequently": 4,
    "Sometimes": 3,
    "Rarely": 2,
    "Not at all": 1
}

i12_health_cols = [col for col in survey_df.columns if col.startswith("i12_health_")]

for col in i12_health_cols:
    survey_df[col] = survey_df[col].map(frequency_map)
    survey_df[col] = survey_df[col].fillna(1)

In [ ]:
# Face mask behaviour scale and binary outcome
mask_items = ["i12_health_1", "i12_health_22", "i12_health_23", "i12_health_25"]
mask_items = [col for col in mask_items if col in survey_df.columns]

survey_df["face_mask_behaviour_scale"] = survey_df[mask_items].median(axis=1)
survey_df["face_mask_behaviour_binary"] = np.where(
    survey_df["face_mask_behaviour_scale"] >= 4, "Yes", "No"
)

# General protective behaviour scale and binary outcome
all_i12_cols = [col for col in survey_df.columns if col.startswith("i12_")]

survey_df["protective_behaviour_scale"] = survey_df[all_i12_cols].median(axis=1)
survey_df["protective_behaviour_binary"] = np.where(
    survey_df["protective_behaviour_scale"] >= 4, "Yes", "No"
)

# Protective behaviour excluding mask-related items
non_mask_items = [col for col in all_i12_cols if col not in mask_items]
survey_df["protective_behaviour_nomask_scale"] = survey_df[non_mask_items].median(axis=1)

In [ ]:
# Recode selected survey variables
# Convert agreement-scale variables into numeric values
agreement_map = {
    "7 - Agree": 7,
    "6": 6,
    "5": 5,
    "4": 4,
    "3": 3,
    "2": 2,
    "1 – Disagree": 1
}

for col in ["r1_1", "r1_2"]:
    if col in survey_df.columns:
        survey_df[col] = survey_df[col].replace(agreement_map)

C:\Users\dukai\AppData\Local\Temp\ipykernel_2340\3696248727.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  survey_df[col] = survey_df[col].replace(agreement_map)


In [31]:
def convert_household_size(value):
    value = str(value)
    if value in [str(i) for i in range(1, 8)]:
        return int(value)
    if value == "8 or more":
        return 8
    if value in ["Prefer not to say", "Don't know", "nan"]:
        return np.nan
    return np.nan

if "household_size" in survey_df.columns:
    survey_df["household_size"] = survey_df["household_size"].apply(convert_household_size)

In [ ]:
# Create comorbidity indicator

d1_cols = [col for col in survey_df.columns if col.startswith("d1_")]

survey_df["d1_comorbidities"] = "Yes"

if "d1_health_99" in survey_df.columns:
    survey_df.loc[survey_df["d1_health_99"] == "Yes", "d1_comorbidities"] = "No"
    survey_df.loc[survey_df["d1_health_99"] == "N/A", "d1_comorbidities"] = "NA"

if "d1_health_98" in survey_df.columns:
    survey_df.loc[survey_df["d1_health_98"] == "Yes", "d1_comorbidities"] = "Prefer_not_to_say"

survey_df = survey_df.drop(columns=d1_cols, errors="ignore")

In [ ]:
# Create time variable and remove unused columns
start_date = survey_df["endtime"].min()
survey_df["week_number"] = ((survey_df["endtime"] - start_date).dt.days // 14) + 1

In [ ]:
# Remove remaining rows with missing values after cleaning
drop_cols = [col for col in ["qweek", "weight"] if col in survey_df.columns]
survey_df = survey_df.drop(columns=drop_cols, errors="ignore")

In [ ]:
key_columns = [
    "endtime",
    "state",
    "face_mask_behaviour_scale",
    "face_mask_behaviour_binary",
    "protective_behaviour_scale",
    "protective_behaviour_binary",
    "household_size",
    "d1_comorbidities"
]

existing_key_columns = [col for col in key_columns if col in survey_df.columns]

# Remove remaining rows with missing values after cleaning
survey_df = survey_df.dropna().copy()

In [ ]:
# Drop original item-level behaviour variables after constructing scales
survey_df = survey_df.drop(columns=i12_health_cols)

In [37]:
survey_df.to_csv("cleaned_data.csv", index=False)

In [38]:
survey_df.columns

Index(['RecordNo', 'endtime', 'i2_health', 'i9_health', 'i11_health', 'age',
       'gender', 'state', 'household_size', 'employment_status', 'WCRex2',
       'cantril_ladder', 'PHQ4_1', 'PHQ4_2', 'PHQ4_3', 'PHQ4_4', 'WCRex1',
       'r1_1', 'r1_2', 'face_mask_behaviour_scale',
       'face_mask_behaviour_binary', 'protective_behaviour_scale',
       'protective_behaviour_binary', 'protective_behaviour_nomask_scale',
       'd1_comorbidities', 'week_number'],
      dtype='object')

In [39]:
cleaned_df = survey_df.copy()

In [40]:
cleaned_df = pd.read_csv("cleaned_data.csv", keep_default_na=False)
mandate_df = pd.read_csv("mandate_start_dates.csv")

# Convert date column directly and create dictionary
mandate_df["mandate_start_date"] = pd.to_datetime(
    mandate_df["mandate_start_date"], format='%Y-%m-%d'
)

states_date = dict(zip(mandate_df["RegionName"], mandate_df["mandate_start_date"]))

# Determine whether each record is within mandate period
def is_within_mandate(row):
    endtime = pd.to_datetime(row['endtime'], format='%Y-%m-%d')
    state = row['state']
    
    # Compare with mandate start date
    return int(states_date[state] <= endtime)


cleaned_df['within_mandate_period'] = cleaned_df.apply(
    is_within_mandate, axis=1
)

# Convert categorical variables to dummy variables
convert_into_dummy_cols = [
    'state', 'gender', 'i9_health', 'employment_status', 'i11_health',
    'WCRex1', 'WCRex2', 'PHQ4_1', 'PHQ4_2', 'PHQ4_3', 'PHQ4_4',
    'd1_comorbidities'
]

# Generate all dummy variables at once
cleaned_df = pd.get_dummies(
    cleaned_df,
    columns=convert_into_dummy_cols,
    drop_first=True
)

cleaned_df.to_csv("cleaned_data_preprocessing.csv", index=False)
